<a href="https://colab.research.google.com/github/dgylayse/AkademiQ_DataScience/blob/main/AkademiQ_Data_Science_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DBSCAN (Density Based Spatial Clustering of Applications with Noise)

eps (epsilon) = Bir noktanın çevresindeki komşuluk yarıçapı

min_samples = Bir bölgenin "yoğun" kabul edilmesi için gereken minimum nokta sayısı

#Nokta Türleri

Core Point= Yoğun bölgenin merkezi

Border Point = Yoğun bölgeye yakın ama yeterince yoğun olmayan nokta

Noise Point = Hiçbir yoğunluğa ait olmayan nokta

In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_excel("Online Retail.xlsx")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [5]:
df = df.dropna(subset=["CustomerID"])

df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      406829 non-null  object        
 8   TotalPrice   406829 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 31.0+ MB


In [8]:
customer_df = df.groupby("CustomerID").agg({
    "InvoiceNo" : "count",
    "Quantity" : "sum",
    "TotalPrice" : "sum"
}).reset_index()

customer_df.columns=[
    "CustomerID", "TotalOrders", "TotalQuantity", "TotalSpent"
]
customer_df.head()

,CustomerID,TotalOrders,TotalQuantity,TotalSpent
0,12346.0,2,0,0.00
1,12347.0,182,2458,4310.00
2,12348.0,31,2341,1797.24
3,12349.0,73,631,1757.55
4,12350.0,17,197,334.40


In [9]:
features = customer_df[["TotalOrders", "TotalQuantity", "TotalSpent"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

In [10]:
scaled_df = pd.DataFrame(X_scaled, columns=features.columns)
scaled_df.head()

,TotalOrders,TotalQuantity,TotalSpent
0,-0.391720,-0.240215,-0.231001
1,0.382657,0.285870,0.293432
2,-0.266959,0.260828,-0.012316
3,-0.086271,-0.105162,-0.017146
4,-0.327188,-0.198051,-0.190312


# K-Distance Plot ile Eps Bulma

In [16]:
from sklearn import neighbors
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

distances = np.sort(distances[:,4])
fig = px.line(
    distances, title="K-Distance Graph ile Eps Seçimi"
)
fig.show()

In [17]:
from sklearn import cluster
dbscan = DBSCAN(
    eps = 0.7,
    min_samples = 5
)

clusters = dbscan.fit_predict(X_scaled)
customer_df["Cluster"] = clusters

#DBSCAN'de -1 etiketi noise anlamına gelir. Fraud user, abnormal sensor, bot activity

customer_df.head()

,CustomerID,TotalOrders,TotalQuantity,TotalSpent,Cluster
0,12346.0,2,0,0.00,0
1,12347.0,182,2458,4310.00,0
2,12348.0,31,2341,1797.24,0
3,12349.0,73,631,1757.55,0
4,12350.0,17,197,334.40,0


In [18]:
print(customer_df["Cluster"].value_counts())

Cluster
 0    4330
-1      42
Name: count, dtype: int64


In [19]:
noise_points = customer_df[customer_df["Cluster"] == -1]
display(noise_points)


,CustomerID,TotalOrders,TotalQuantity,TotalSpent,Cluster
55,12415.0,778,77242,123725.45,-1
330,12748.0,4642,24210,29072.10,-1
436,12901.0,125,20915,16293.10,-1
458,12931.0,102,23377,33462.81,-1
525,13027.0,26,17280,6912.00,-1
564,13081.0,1061,19021,27964.48,-1
568,13089.0,1857,30787,57385.88,-1
576,13098.0,605,15911,28658.88,-1
698,13263.0,1677,4780,7454.07,-1
803,13408.0,501,16119,27487.41,-1


In [22]:
fig = px.scatter(
    customer_df,
    x="TotalSpent",
    y="TotalQuantity",
    color="Cluster",
    title="DBSCAN ile Müşteri Segmentasyonu"
)
fig.show()

In [24]:
fig = px.scatter_3d(
    customer_df,
    x="TotalSpent",
    y="TotalQuantity",
    z="TotalOrders",
    color="Cluster",
    title="DBSCAN ile Müşteri Segmentasyonu"
)

fig.show()

In [25]:
customer_df.groupby("Cluster").mean(numeric_only=True)

,CustomerID,TotalOrders,TotalQuantity,TotalSpent
Cluster,,,,
-1,15144.119048,1166.928571,33430.309524,57413.067143
0,15301.186605,82.636952,808.964203,1359.980830


# PCA ile Görselleştirme

In [26]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df.head()

,PC1,PC2
0,-0.475327,-0.196386
1,0.539155,0.153233
2,0.034853,-0.314594
3,-0.116477,-0.038248
4,-0.393735,-0.165722
